# 아기 감지 OD 학습 (Colab 래퍼)

**OD_Training_Standard 진입점을 그대로 호출하는 얇은 래퍼**입니다. 학습 로직은 전부 표준 쪽에 있습니다.

전제 (Google Drive `내 드라이브/BabyMon/` 에 업로드):
- `BabyMon/Training_Standard/` — `D:\Code\Training_Standard` 저장소 폴더 통째로
- `BabyMon/yolo_baby/` — 데이터셋 (`images/`, `labels/`; 빈 txt = 네거티브)
- `BabyMon/yolo_baby/params.json` — 이 폴더의 `params.json` 을 업로드 (하이퍼파라미터)

실행: 런타임 → **T4 GPU** 설정 후 셀 순서대로.
로컬(GTX 1650)에서도 같은 명령을 `run_training.py` 로 실행할 수 있습니다 (README 참고).

In [ ]:
# ① 패키지 설치 (Colab 기본 torch 사용)
!pip install -q "ultralytics>=8.3.0" onnx onnxruntime

In [ ]:
# ② Drive 마운트 + 경로/사전 점검
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE   = Path('/content/drive/MyDrive/BabyMon')
TS_ROOT = DRIVE / 'Training_Standard'
DATASET = DRIVE / 'yolo_baby'
PARAMS  = DATASET / 'params.json'
ENTRY   = TS_ROOT / 'Training Standard' / 'OD_Training_Standard' / 'od_training_standard_local.py'

for p, name in [(TS_ROOT, 'Training_Standard 저장소'), (DATASET, 'yolo_baby 데이터셋'),
                (PARAMS, 'params.json'), (ENTRY, 'OD 학습 진입점')]:
    print('O' if p.exists() else 'X (없음)', name, '->', p)

In [ ]:
# ③ 학습 — OD_Training_Standard 진입점 실행 (로컬 run_training.py 와 동일 호출)
import os, subprocess, sys

assert ENTRY.is_file(), f'학습 진입점이 없습니다: {ENTRY} — Drive 업로드를 확인하세요'
env = os.environ.copy()
env['PYTHONPATH'] = f"{TS_ROOT}:{ENTRY.parent}"
ret = subprocess.call([sys.executable, str(ENTRY),
                       '--dataset_path', str(DATASET),
                       '--params', str(PARAMS)],
                      cwd=str(ENTRY.parent), env=env)
print('종료 코드:', ret)
assert ret == 0, '학습 실패 — 위 로그를 확인하세요'

In [ ]:
# ④ 산출물 확인 + best.onnx export (RDK X5 / BPU 용: opset 11, imgsz 640, batch 1)
import os, shutil, torch
from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel

cands = sorted(ENTRY.parent.rglob('best/model.pth'), key=os.path.getmtime, reverse=True)
assert cands, f'산출물(best/model.pth)을 찾지 못했습니다: {ENTRY.parent} 아래를 확인하세요'
MODEL = cands[0]
print('산출물:', MODEL)

ckpt = torch.load(MODEL, map_location='cpu', weights_only=False)
if isinstance(ckpt, dict) and hasattr(ckpt.get('model'), 'yaml'):
    yolo = YOLO(str(MODEL))
else:  # state_dict 폴백 산출물 -> yolo11n 재구성 (단일 클래스 baby)
    state = ckpt.get('state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    dm = DetectionModel(cfg='yolo11n.yaml', ch=3, nc=1, verbose=False)
    try:
        dm.load_state_dict(state)
    except RuntimeError:  # 'model.'/'module.' 접두어 1단계 제거 후 재시도
        dm.load_state_dict({k.split('.', 1)[1]: v for k, v in state.items() if '.' in k})
    dm.names = {0: 'baby'}
    yolo = YOLO('yolo11n.yaml', task='detect')
    yolo.model = dm

onnx_path = yolo.export(format='onnx', opset=11, imgsz=640, batch=1, simplify=True, dynamic=False)
out = DATASET / 'export' / 'best.onnx'
out.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(onnx_path), str(out))
print('export 완료:', out)
print('다음: hb_mapper makertbin (march bayes-e, input_type_train: bgr — 학습 채널 BGR) -> best.bin -> RDK X5')